In [13]:
# --- Third-party libraries: numerical + plotting ---
import numpy as np                               # numerical computing (arrays, arange, hstack, etc.)

# --- Deep learning framework ---
import tensorflow as tf                          # core TensorFlow library
import tensorflow.keras as keras                 # high-level neural network API

### 1. MNIST laden

In [14]:
tf.random.set_seed(42)
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Augabe der Shape: Anzahl Bilder, Höhe, Breite 
print("Train shape:", x_train.shape)

Train shape: (60000, 28, 28)


### 2. Preprocessing

#### 2.1 Normalisierung

In [15]:
# Werte vom ursprünglichen Bereich 0–255 in den Bereich 0–1 umwandeln.
x_train = x_train / 255.0
x_test = x_test / 255.0

#### 2.2 Kanal-Dimension

In [16]:
# CNN um eine Kanal-Dimension erweitern um die geforderte Form (height, width, channels) zu erreichen
# Shape: (Anzahl Bilder, Höhe, Breite) zu (Anzahl Bilder, Höhe, Breite, Kanäle)

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

#### 2.3 Daten Augmentation

In [17]:
# Daten leicht rotieren und verschieben, um Variationen zu simulieren
data_augmentation = keras.Sequential([
    keras.layers.RandomRotation(0.05),
    keras.layers.RandomTranslation(0.05, 0.05)
])

### 3. CNN Modell

In [18]:
model = keras.Sequential([

    # ------------------------------------------------------------
    # Input layer
    # Erwartet Bilder der Form (28, 28, 1)
    # ------------------------------------------------------------
    keras.layers.Input(shape=(28,28,1)),

    # ------------------------------------------------------------
    # Implementierung der Data Augmentation während des Trainings
    # um Robustheit gegenüber Rotation und Verschiebung zu lernen.
    # ------------------------------------------------------------
    data_augmentation,


    # ------------------------------------------------------------
    # Block 1: Low-level Feature Extraction
    # Lernen einfacher Muster wie Kanten und Linien
    # ------------------------------------------------------------

    keras.layers.Conv2D(64, (3,3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.Conv2D(64, (3,3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.MaxPooling2D((2,2)),

    keras.layers.Dropout(0.2),


    # ------------------------------------------------------------
    # Block 2: High-level Feature Extraction
    # Lernen komplexerer Strukturen
    # ------------------------------------------------------------

    keras.layers.Conv2D(64, (3,3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    
    keras.layers.Conv2D(64, (3,3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.MaxPooling2D((2,2)),

    keras.layers.Dropout(0.3),


    # ------------------------------------------------------------
    # Classification Head
    # Umwandeln von Feature Maps in Klassenentscheidung
    # ------------------------------------------------------------

    keras.layers.Flatten(),

    keras.layers.Dense(128, activation="relu"),

    keras.layers.Dropout(0.4),

    keras.layers.Dense(10, activation="softmax")
])


### 4. Kompilieren

In [19]:
# Modell kompilieren
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential (Sequential)     (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 28, 28, 64)        640       
                                                                 
 batch_normalization (Batch  (None, 28, 28, 64)        256       
 Normalization)                                                  
                                                                 
 activation (Activation)     (None, 28, 28, 64)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 28, 28, 64)        36928     
                                                                 
 batch_normalization_1 (Bat  (None, 28, 28, 64)        256       
 chNormalization)                                     

### 5. Training

In [20]:
# Abbruchkriterium festlegen um Overfitting zu verhindern und Trainingszeit zu sparen
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [21]:
# Modell auf Trainingsdaten trainieren und Validierung überwachen
model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/20
1688/1688 [==============================] - 164s 96ms/step - loss: 0.3712 - accuracy: 0.8855 - val_loss: 0.0625 - val_accuracy: 0.9825
Epoch 2/20
1688/1688 [==============================] - 159s 94ms/step - loss: 0.1648 - accuracy: 0.9521 - val_loss: 0.0432 - val_accuracy: 0.9877
Epoch 3/20
1688/1688 [==============================] - 156s 92ms/step - loss: 0.1338 - accuracy: 0.9608 - val_loss: 0.0400 - val_accuracy: 0.9892
Epoch 4/20
1688/1688 [==============================] - 158s 94ms/step - loss: 0.1044 - accuracy: 0.9697 - val_loss: 0.0328 - val_accuracy: 0.9923
Epoch 5/20
1688/1688 [==============================] - 160s 95ms/step - loss: 0.0943 - accuracy: 0.9730 - val_loss: 0.0304 - val_accuracy: 0.9923
Epoch 6/20
1688/1688 [==============================] - 160s 95ms/step - loss: 0.0797 - accuracy: 0.9771 - val_loss: 0.0274 - val_accuracy: 0.9945
Epoch 7/20
1688/1688 [==============================] - 158s 94ms/step - loss: 0.0748 - accuracy: 0.9786 - val_loss: 0

### 6. Modell speichern

In [22]:
model.save("CNN_Vincent.keras")

### 7. Evaluation

In [23]:
# Modell auf Testdaten evaluieren
loss, accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", accuracy)

313/313 [==============================] - 6s 18ms/step - loss: 0.0169 - accuracy: 0.9947
Test Accuracy: 0.994700014591217


### 8. Nachweis: Grid-Search